# 11a — Demographic Baseline Models

This notebook runs the shared Week 4 baseline modeling framework using **demographic features only**.

## Objective

The notebook:

- loads the predefined participant-level train, validation, and test datasets;
- selects only the demographic modality;
- verifies the selected feature structure and class distributions;
- uses the shared preprocessing, model, workflow, evaluation, and output utilities from `src/modeling`;
- evaluates the Dummy Classifier, multinomial Logistic Regression, and Decision Tree;
- uses the project's configured class weighting for Logistic Regression and Decision Tree;
- reports validation and test metrics, classification reports, and confusion matrices;
- saves model outputs using the shared functions in `src/modeling/outputs.py`.

No model training, preprocessing, or evaluation logic is reimplemented here. The notebook uses the shared framework exactly as provided.

## 1. Libraries and configuration

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

# Support execution from either the project root or the notebooks directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 2. Import the shared modeling framework

The demographic experiment uses the same project-wide data loading, split validation, preprocessing, baseline model constructors, modeling workflow, evaluation, and output utilities used by the shared framework.

In [ ]:
from src.modeling.data_loader import load_all_datasets

from src.modeling.split_validation import validate_split

from src.modeling.preprocessing import (
    prepare_dataset,
    identify_feature_types,
)

from src.modeling.baseline import get_dummy_classifier

from src.modeling.models import (
    get_logistic_regression,
    get_decision_tree,
)

from src.modeling.workflow import run_models

from src.modeling.outputs import (
    save_metrics,
    save_classification_report,
    save_confusion_matrix,
    save_model_comparison,
    save_table,
)

from src.modeling.config import (
    TARGET_COLUMN,
    CLASS_WEIGHT,
    USE_CLASS_WEIGHT,
    PRIMARY_METRIC,
)

## 3. Load participant-level datasets

In [ ]:
train_df, validation_df, test_df = load_all_datasets()

print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {validation_df.shape}")
print(f"Test shape:       {test_df.shape}")

The predefined participant-level splits are used rather than creating new splits inside this notebook. This keeps the demographic experiment consistent with the shared modeling framework.

## 4. Validate the participant-level split

Before modeling, verify that the predefined train, validation, and test datasets remain participant-independent and review the participant counts and class distributions.

In [ ]:
split_summary = validate_split(
    train_df,
    validation_df,
    test_df,
)

split_summary

## 5. Select demographic-only features

In [ ]:
demographic_features = [
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
    "gender",
    "handedness",
    "family_history_any",
    "family_history_first_degree",
    "alcohol_effect_on_tremor",
]

# Retain the fields required by the shared framework in addition to
# the demographic predictors. Identifier and diagnosis-related fields
# are removed automatically by the shared preprocessing pipeline.
framework_columns = [
    "patient_id",
    "study_id",
    "duplicate_patient_id",
    "label",
    "condition_original",
    "condition_group",
]

required_demographic_columns = demographic_features + ["patient_id", TARGET_COLUMN]

for split_name, dataframe in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    missing = [
        column
        for column in required_demographic_columns
        if column not in dataframe.columns
    ]
    assert not missing, (
        f"{split_name} dataset is missing required demographic columns: {missing}"
    )

columns_to_keep = [
    column
    for column in framework_columns + demographic_features
    if column in train_df.columns
]

train_demo = train_df[columns_to_keep].copy()
validation_demo = validation_df[columns_to_keep].copy()
test_demo = test_df[columns_to_keep].copy()

print(f"Demographic train shape:      {train_demo.shape}")
print(f"Demographic validation shape: {validation_demo.shape}")
print(f"Demographic test shape:       {test_demo.shape}")

print("\nColumns retained for the demographic experiment:")
print(train_demo.columns.tolist())

The modeling dataset is restricted to the cleaned demographic variables produced by the demographic preparation workflow. Diagnosis descriptors and identifiers are retained only where needed by the shared framework and are excluded from the predictor matrix during preprocessing.

## 6. Review demographic feature quality

In [ ]:
class_distribution = pd.DataFrame({
    "Train": train_demo[TARGET_COLUMN].value_counts().sort_index(),
    "Validation": validation_demo[TARGET_COLUMN].value_counts().sort_index(),
    "Test": test_demo[TARGET_COLUMN].value_counts().sort_index(),
}).fillna(0).astype(int)

class_distribution

In [ ]:
missing_summary = pd.DataFrame({
    "Train": train_demo[demographic_features].isna().sum(),
    "Validation": validation_demo[demographic_features].isna().sum(),
    "Test": test_demo[demographic_features].isna().sum(),
}).sort_values("Train", ascending=False)

missing_summary

Missing demographic values are not manually filled in this notebook. The shared preprocessing pipeline handles numeric and categorical missingness during model fitting.

## 7. Verify preprocessing and leakage prevention

In [ ]:
X_train, y_train, preprocessing = prepare_dataset(
    train_demo,
)

numerical_features, categorical_features = identify_feature_types(
    X_train
)

feature_summary = pd.DataFrame({
    "Feature Type": [
        "Numerical",
        "Categorical",
    ],
    "Count": [
        len(numerical_features),
        len(categorical_features),
    ],
})

print("Predictor columns:")
print(X_train.columns.tolist())

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

display(feature_summary)

# Confirm that identifiers, target labels, and diagnosis descriptors
# are not present in the predictor matrix.
leakage_columns = [
    "patient_id",
    "study_id",
    "duplicate_patient_id",
    "label",
    "condition_original",
    "condition_group",
]

remaining_leakage_columns = [
    column
    for column in leakage_columns
    if column in X_train.columns
]

assert not remaining_leakage_columns, (
    f"Leakage-related columns remain in X_train: {remaining_leakage_columns}"
)

print("Leakage prevention check: PASS")

In [ ]:
preprocessing

The shared preprocessing pipeline applies median imputation and standardization to numerical variables, and most-frequent imputation with one-hot encoding to categorical variables. Preprocessing is fitted inside the modeling pipeline rather than being performed manually in this notebook.

## 8. Define demographic baseline models

In [ ]:
models = {
    "Dummy": get_dummy_classifier(),
    "Logistic Regression": get_logistic_regression(),
    "Decision Tree": get_decision_tree(),
}

print("Class weighting enabled:", USE_CLASS_WEIGHT)
print("Configured class weight:", CLASS_WEIGHT)
print("Primary evaluation metric:", PRIMARY_METRIC)

models

The Dummy Classifier provides the reference baseline. Logistic Regression and Decision Tree use the class-weight setting defined centrally in `src/modeling/config.py`. No model-specific settings are overridden in this notebook.

## 9. Run demographic baseline models

In [ ]:
results = run_models(
    models=models,
    train_df=train_demo,
    validation_df=validation_demo,
    test_df=test_demo,
)

print("Completed models:")
print(list(results.keys()))

## 10. Compare validation and test performance

In [ ]:
validation_comparison = pd.DataFrame({
    name: result["metrics"]
    for name, result in results.items()
}).T

validation_comparison = validation_comparison.sort_values(
    PRIMARY_METRIC,
    ascending=False,
)

validation_comparison

In [ ]:
test_comparison = pd.DataFrame({
    name: result["test_metrics"]
    for name, result in results.items()
}).T

test_comparison = test_comparison.sort_values(
    PRIMARY_METRIC,
    ascending=False,
)

test_comparison

### 10.1 Classification reports

In [ ]:
for model_name, result in results.items():
    print("=" * 70)
    print(f"{model_name.upper()} — VALIDATION CLASSIFICATION REPORT")
    print("=" * 70)
    display(result["classification_report"])

    print(f"\n{model_name.upper()} — TEST CLASSIFICATION REPORT")
    display(result["test_report"])
    print()

### 10.2 Confusion matrices

In [ ]:
for model_name, result in results.items():
    print("=" * 70)
    print(f"{model_name.upper()} — VALIDATION CONFUSION MATRIX")
    print("=" * 70)
    display(result["confusion_matrix"])

    print(f"\n{model_name.upper()} — TEST CONFUSION MATRIX")
    display(result["test_confusion_matrix"])
    print()

## 11. Save modeling outputs

All demographic model outputs are saved through the shared functions in `src/modeling/outputs.py`. Filenames are prefixed with `demographics_` so they remain distinct from questionnaire and wearable experiments.

In [ ]:
model_filename_map = {
    "Dummy": "dummy",
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
}

for model_name, result in results.items():
    filename = model_filename_map[model_name]

    # Validation outputs
    save_metrics(
        result["metrics"],
        f"demographics_{filename}_validation_metrics.csv",
    )

    save_classification_report(
        result["classification_report"],
        f"demographics_{filename}_validation_classification_report.csv",
    )

    save_confusion_matrix(
        result["confusion_matrix"],
        f"demographics_{filename}_validation_confusion_matrix.csv",
    )

    # Test outputs
    save_metrics(
        result["test_metrics"],
        f"demographics_{filename}_test_metrics.csv",
    )

    save_classification_report(
        result["test_report"],
        f"demographics_{filename}_test_classification_report.csv",
    )

    save_confusion_matrix(
        result["test_confusion_matrix"],
        f"demographics_{filename}_test_confusion_matrix.csv",
    )

save_model_comparison(
    validation_comparison,
    "demographics_validation_model_comparison.csv",
)

save_model_comparison(
    test_comparison,
    "demographics_test_model_comparison.csv",
)

save_table(
    class_distribution.reset_index().rename(columns={"index": "label"}),
    "demographics_class_distribution.csv",
)

save_table(
    missing_summary.reset_index().rename(columns={"index": "feature"}),
    "demographics_missing_value_summary.csv",
)

print("Demographic modeling outputs saved successfully.")

## 12. Initial observations

In [ ]:
best_validation_model = validation_comparison.index[0]
best_validation_score = validation_comparison.iloc[0][PRIMARY_METRIC]

best_test_model = test_comparison.index[0]
best_test_score = test_comparison.iloc[0][PRIMARY_METRIC]

dummy_validation_score = validation_comparison.loc[
    "Dummy",
    PRIMARY_METRIC,
]

print("INITIAL DEMOGRAPHIC BASELINE OBSERVATIONS")
print("-" * 45)
print(
    f"Best validation model: {best_validation_model} "
    f"({PRIMARY_METRIC} = {best_validation_score:.3f})"
)
print(
    f"Best test model: {best_test_model} "
    f"({PRIMARY_METRIC} = {best_test_score:.3f})"
)
print(
    f"Dummy validation {PRIMARY_METRIC}: "
    f"{dummy_validation_score:.3f}"
)

if best_validation_score > dummy_validation_score:
    print(
        "At least one demographic model outperformed the Dummy "
        "Classifier on the primary validation metric."
    )
else:
    print(
        "The demographic models did not outperform the Dummy "
        "Classifier on the primary validation metric."
    )

print(
    "Use the class-level reports and confusion matrices above to "
    "describe which diagnostic groups are easiest or hardest to classify."
)

## 13. Final validation status

In [ ]:
validation_status = pd.DataFrame({
    "Requirement": [
        "Predefined participant splits loaded",
        "Participant split validated",
        "Demographic-only predictors selected",
        "Leakage-related columns excluded by shared preprocessing",
        "Dummy Classifier evaluated",
        "Multinomial Logistic Regression evaluated",
        "Decision Tree evaluated",
        "Configured class weighting used where applicable",
        "Validation metrics produced",
        "Test metrics produced",
        "Classification reports produced",
        "Confusion matrices produced",
        "Shared output functions used",
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
    ],
})

validation_status

## 14. Conclusion

After all cells run successfully, this notebook demonstrates the demographic-only baseline modeling workflow using the project's shared modeling framework.

A successful final execution means the following can be confirmed:

1. The predefined participant-level train, validation, and test splits were used.
2. Only demographic predictors were supplied to the shared modeling workflow.
3. Identifier and diagnosis-related columns were excluded from model predictors.
4. The Dummy Classifier, Logistic Regression, and Decision Tree were evaluated without rewriting shared training or evaluation code.
5. The project's configured class weighting was used for applicable models.
6. Validation and test metrics, classification reports, and confusion matrices were produced.
7. Demographic-specific output files were saved through the shared output utilities.
8. Initial observations can be based directly on the generated model comparison tables and class-level results.